# DS-Harness — audit indépendant reproductible

Agents immuables ; expériences CPU mesurées ; expériences GPU explicitement non exécutées. Lire REPORT.md pour le rapport initial et la hiérarchie des hypothèses. Aucun score privé inventé. Les programmes des contre-exemples sont écrits à la main, pas générés par Qwen.

## 00 — Setup et reproductibilité
Exécuter depuis ce dossier, projects, ou la racine du workspace. RERUN_CPU rejoue les mesures ; RUN_GPU lance une expérience hors budget de génération et replay, sur Linux/CUDA seulement. Aucun accès au leaderboard.

In [1]:
from pathlib import Path
import sys, json, hashlib, platform
import pandas as pd
candidates=[Path.cwd(),Path.cwd()/'audit_independent',Path.cwd()/'projects/audit_independent']
AUDIT=next(p for p in candidates if (p/'audit.py').exists()).resolve()
sys.path.insert(0,str(AUDIT)); sys.path.insert(0,str(AUDIT.parent))
from audit import inventory, reproduce, OUT, HANDOFF, FILES
RERUN_CPU=False
RUN_GPU=False
print('Python',sys.version)
print('Platform',platform.platform())
print('Inputs',HANDOFF)


Python 3.12.13 (main, Jun 23 2026, 15:23:43) [MSC v.1944 64 bit (AMD64)]
Platform Windows-11-10.0.26200-SP0
Inputs C:\Users\GOUA\Desktop\COURS MS2A\2026-2027\ML EN PRATIQUE\projects\CODEX_AUDIT_HANDOFF


## 01 — Artefacts et hashes
MEASURED : comparaison des octets locaux au manifeste fourni. L’identité avec une soumission téléchargée demeure une question distincte.

In [2]:
manifest,changes=inventory()
print(pd.DataFrame(manifest).to_string(index=False))
print(pd.DataFrame(changes).query("symbol != 'Agent'").to_string(index=False))


                          file  bytes                                                           sha256
         CODEX_AUDIT_PROMPT.md  10823 9cf028d2bfa9da7e806ef318e73f3a55ded26276ea1cb01bc93f72320660ae6a
                      dev.json 923286 7e06aeeafa086bc7e93180e9f66073be3569c32ef073df81f9c56824515ee289
               dsh_official.py  15301 391b22c491c25d276adacb6162ff86a65f968a1273f8d816a3d158210ac27a7c
                     README.md   2122 358abad5ee0a35b78200c709651152f18e300e3e92b30a0a2f8e6e4979b42eac
      STAGE3_SUBMITTED_92_5.py 130431 00bfd0d2ef05c055cc53c8814cc317ec05f3d10c08dfdedb14ddd6ff30e93aa7
       TEST1_SUBMITTED_86_7.py 147257 a83da219d907b327b5fe76b28b3b769f429b41e7b8ad08815800c931753bdf28
TEST2_MINIMAL_NOT_SUBMITTED.py 148416 71ff3b5cdde4d91686d3f0d77b4fe3f6f99550b20899f76888eb2893d9a2f29e
       TEST2_SUBMITTED_77_5.py 200878 aa3a2fd0af579bada2efff0fcc4c7cc379fbccb397a4633bb2550b972db7a66e
         V10_SUBMITTED_81_7.py 152153 0f5491050434ec2b71c0dc72b676f8d77e1

## 02 — Reproduction du public
Hypothèse : les Tables publiques ne traversent pas le fallback. Méthode : vrai solve, modèle remplacé par un témoin qui signale une demande de génération. Les réponses par défaut ne sont pas comptées comme règles réussies.

In [3]:
if RERUN_CPU:
    reproduce()
print(pd.read_csv(OUT/'dev_summary.csv').to_string(index=False))


  agent family  total  covered  correct
 stage3   calc     65       25       25
 stage3  table     80       80       80
 stage3    fit     35       35       35
  test1   calc     65       25       25
  test1  table     80       80       80
  test1    fit     35       35       35
  test2   calc     65       65       65
  test2  table     80       80       80
  test2    fit     35       35       35
minimal   calc     65       25       25
minimal  table     80       80       80
minimal    fit     35       35       35
    v10   calc     65       65       65
    v10  table     80       80       80
    v10    fit     35       35       35


## 03 — Cartographie règle / fallback et fidélité de l’évaluateur
Les appels aux règles sont conservés par tâche dans real_solve_dev.csv. Comparaison à l’ancien évaluateur : les 40 écarts de couverture TEST2 sont des No-files, pas des Tables.

In [4]:
real=pd.read_csv(OUT/'real_solve_dev.csv')
legacy=pd.read_csv(OUT/'legacy_dev.csv')
paired=real.merge(legacy,on=['agent','id'],suffixes=('_solve','_legacy'))
diff=paired[paired.covered_solve != paired.covered_legacy]
print(diff.groupby(['agent','family_solve']).size().to_string())
print(real[['agent','id','calls']].head().to_string(index=False))


agent  family_solve
test2  calc            40
 agent    id                                                                                                                                                  calls
stage3 d_001 [["table_stat_rule", "None"], ["table_group_rule", "None"], ["table_join_rule", "None"], ["table_derive_rule", "None"], ["table_filter_rule", "23.5"]]
stage3 d_002                                                                                                                   [["fit_rules", "0.908888119952154"]]
stage3 d_003                                                                                                                  [["fit_rules", "242.64372117542464"]]
stage3 d_004                                                                                                                  [["fit_rules", "0.5188268379322779"]]
stage3 d_005                                                               [["table_stat_rule", "None"], ["table_group_rule", "None"],

## 04 — Forced fallback Tables
NON EXÉCUTÉ avec Qwen sur cet hôte : CUDA absent et aucun poids Qwen en cache. Le module gpu_experiments prépare les 80 prompts avec le prétraitement Stage3 et appelle Agent.generate, y compris son préfixe Python. Cette expérience isole la génération hors délai ; elle ne remplace pas un solve complet en lots mixtes.

In [5]:
import torch
from huggingface_hub import constants
print('torch',torch.__version__,'CUDA',torch.cuda.is_available())
print('Cache models',[p.name for p in Path(constants.HF_HUB_CACHE).glob('models--*')])
from gpu_experiments import generate_cache,replay_cache,summarize_replay
cache=None
if RUN_GPU:
    cache=generate_cache()
else:
    print('NOT RUN: forced fallback Qwen')


torch 2.14.1+cpu CUDA False
Cache models ['models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2']
NOT RUN: forced fallback Qwen


## 05 — Cache des générations
La clé inclut modèle, révision, seed, hash agent, hash dev et paramètres de génération. Chaque ligne conserve prompt, texte et code. Le replay ne régénère jamais. Les fichiers partiels portent .partial.json ; ils ne doivent pas être utilisés comme résultats complets.

In [6]:
print('Caches complets disponibles:',[p.name for p in OUT.glob('generations_*.json') if '.partial.' not in p.name])

Caches complets disponibles: []


## 06 — Exécuteurs A/B/C : contre-exemples et ablation
MEASURED : collision pd et compilation collective. Windows utilise un shim POSIX uniquement pour les programmes finis ; aucune inférence de timeout/mémoire Linux. L’ablation de portée exécute le vrai solve avec code fixe et budgets relâchés.

In [7]:
if RERUN_CPU:
    import probes,ablation
    probes.run(); ablation.run()
probes_df=pd.read_csv(OUT/'executor_probes.csv')
print(probes_df[['agent','probe','result']].to_string(index=False))
ab=pd.read_csv(OUT/'wrapper_ablation.csv')
print(ab[['agent','wrapper_alias_fix','answer','correct']].to_string(index=False))


  agent                          probe                                                                                                                                                                                                               result
 stage3                        control                                                                                                                                                                                                                  6.0
 stage3                  import_pandas                                                                                                                                                                                                                  6.0
 stage3                   import_numpy                                                                                                                                                                                                              

## 07 — Replay réel et modèle Coder vs Qwen3
À exécuter sous Linux/CUDA : créer deux caches, changer seulement model. Même prompt Stage3, 500 tokens, greedy + quatre samples. Séparer l’effet modèle de l’effet prompt. La comparaison reste hors budget ; les temps seront à mesurer dans les lots officiels ensuite.

In [8]:
if RUN_GPU and cache:
    replay=replay_cache(cache)
    print(summarize_replay(replay).to_string())
    # Deuxième bras séparé, à lancer quand le premier est inspecté :
    # qwen3_cache=generate_cache(model='Qwen/Qwen3-1.7B')
else:
    print('NOT RUN: executor replay on Qwen cache / model comparison')


NOT RUN: executor replay on Qwen cache / model comparison


## 08 — Greedy, samples, oracle et vote
Le replay calcule oracle_samples_at4 (les quatre samples seulement), oracle_greedy_plus4 et vote_accuracy séparément. Ne pas appeler cinq programmes « oracle@4 ». Un écart oracle-vote est une borne de potentiel, pas un gain réalisable garanti. Résultats Qwen : non disponibles.

## 09 — Repair
Expérience restante : journaliser le vrai solve sous Linux, avec puis sans DSH_NO_REPAIR, mêmes tâches, seeds et budget. Mesurer les transitions correct/faux et secondes par gain net ; les erreurs du wrapper doivent être corrigées avant de juger le modèle. Le témoin pd a montré un repair inutile, sans mesurer son rendement sur Qwen.

## 10 — Prompts
Stage3 et TEST1 ont les mêmes fonctions/prompts initiaux selon le diff. Une ablation du prompt impose de nouvelles générations, puis un replay identique ; conserver le même code empêcherait de mesurer l’effet du prompt.

## 11 — Shadow rules
Expérience restante : TEST2 original versus extensions désactivées en mémoire, puis ablation distincte des garde-fous stat/filter. Conserver suggestions, décisions et désaccords par tâche. Sur le dev actuel, les règles historiques absorbent déjà toutes les Tables : ce corpus seul ne mesure pas la valeur des extensions privées.

## 12 — No-files et Fits
MEASURED : 65/65 No-files déterministes TEST2, contre 25/65 Stage3/TEST1 ; 35/35 Fits publics partout. L’égalité des agrégats privés ne permet aucune conclusion sur l’identité des erreurs. Aucun changement Fits en production proposé.

## 13 — Robustesse et coût par lot
Expérience Linux restante : 1, 4, 8 fallbacks, lots mixtes, permutations d’ordre, limites 40 s / RAM. Les timeouts collectifs peuvent perdre les résultats déjà calculés. Le coût d’une tâche influence les sept autres ; les mesures tâche par tâche hors budget ne suffisent pas.

## 14 — Construction du challenger
Aucun agent de production modifié. L’alias pandas corrigé n’existe que dans le transport expérimental ablation.py. Candidat de contrôle proposé : règles et prompts Stage3, exécuteur individuel conforme, puis batch après validation du contrat et du budget. Pas de nouvelle règle avant comparaison au fallback.

## 15 — Rapport GO / NO-GO
NO-GO pour soumission privée. Défauts d’exécuteur prouvés ; incidence privée, oracle Qwen, repair, p95 GPU et capacité de généralisation non mesurés. Lire le rapport détaillé ci-dessous.

In [9]:
print((AUDIT/'REPORT.md').read_text(encoding='utf-8'))

# Audit indépendant — premier rapport expérimental

Les agents de référence n'ont pas été modifiés. Aucune soumission privée effectuée.

## Conclusion principale

**MEASURED : un défaut de portée Python casse les programmes qui importent pandas dans les exécuteurs groupés.** Ce défaut existe dans les samples TEST1 et TEST2 minimal ; TEST2 large l'expose aussi au greedy, aux continuations et au repair. Il fournit une explication alternative concrète à l'effondrement Tables. Sa contribution au score privé reste **HYPOTHESIS**, faute de générations et de résultats privés par tâche.

## 1. Provenance

Les sept tailles et SHA-256 annoncés (cinq agents, helper, dev) correspondent exactement aux octets du package local. Voir `results/manifest.csv` pour les empreintes complètes. Cela authentifie le package reçu, pas son identité avec un téléchargement ML-Arena indépendant.

| Artefact | Octets | SHA-256 (préfixe) | Provenance externe |
|---|---:|---|---|
| Stage3 | 130431 | 00bfd0d2ef05 | décl